## Errata pós-auditoria — Etapa 5.6

Resultados e saídas abaixo são históricos e não foram reexecutados nesta revisão. A MLP supera a maioria, mas não demonstrou ganho consistente sobre tabela retrospectiva de frequências (Accuracy 69,28%; Macro F1 67,27%; AUC 0,7389 versus MLP 68,82%; 67,09%; 0,7412). A tabela foi criada após o teste, sem caráter de avaliação independente. AUC mede ordenação, não acurácia/calibração.

A rejeição histórica de Platt por recall confundiu calibração com ponto de operação: 0,5 calibrado equivalia a aproximadamente 0,5218 original. O ganho de Brier/Log Loss/ECE de 2024 não é refutado pela queda do recall nesse limiar. Mantidos sigmoid não calibrada e limiar 0,5; nenhuma alteração de modelo. Em 2025 houve superestimação média de roubo de aproximadamente 8,10 pp, com AUC semelhante.

Novas integrações: `final5.validation.SafeClassifier`, contrato 1.1.0. Mês/dia ausentes e categorias desconhecidas bloqueados. Ver [correções](../reports/relatorio_correcoes_pos_auditoria.md) e [limitações](../docs/limitacoes_modelo_final.md).

2025 foi explorado estatisticamente na Etapa 1, avaliado na Etapa 5 e reaberto na auditoria. Não é teste novo para v2. Git próprio iniciado só pós-auditoria. Não executar treinamentos, nova seleção ou avaliação de 2025 por este notebook.


# Etapa5 — Avaliação final da Dense64

Classificação de BOs de celulares subtraídos em São Paulo capital:0=FURTO,1=ROUBO. Dense32/64/64→32 foram comparadas na Etapa4. Dense64, semente42, foi fixada antes do teste por estabilidade e simplicidade relativa. O notebook lê os resultados da única avaliação final de2025, sem treinar, selecionar ou reinferir.

In [1]:
from pathlib import Path
import json,csv
ROOT=Path.cwd()
if not (ROOT/'models/final').exists():ROOT=ROOT.parent
OUT=ROOT/'reports/etapa5'
manifest=json.loads((ROOT/'models/final/manifesto.json').read_text(encoding='utf-8'))
print('Versão:',manifest['model_id'],'Congelamento UTC:',manifest['frozen_at'])
print(manifest['architecture'],manifest['features_order'])
print('Hashes:',manifest['frozen_files'])


Versão: gs-celulares-sp-dense64-v1.0.0 Congelamento UTC: 2026-11-09T22:42:06.159595+00:00
{'input': 299, 'hidden': [64], 'activations': ['relu', 'sigmoid'], 'parameters': 19265} ['mes', 'dia_semana', 'tipo_local', 'bairro']
Hashes: {'modelo.keras': 'dadad1326e6ae58d0f26f7e69638a9283d46863c9ecb7d56e76e037b4940091c', 'preprocessador.joblib': 'b8b3cd3662505b872b5e6d34296fba53a5b1ce5e065811d128f0423aa050a524', 'config_final.json': '235867fff828c3f7569c21abe4f8df619740ca210794c9981159db4a96d0bb1a', 'contrato.json': 'c328e0496c4c4a8c2efd51292002a59a3160d09dfa58c83930f6febad58e111b'}


## Decisão antes do teste

Artefato B64_full_s42, 299 entradas,64 ReLU→1 sigmoid,19.265 parâmetros, Adam0,001, BCE, batch256,3 épocas, threshold0,5, sem dropout/L2. O pré-processador original usa frequência50 e alias de local. Platt foi ajustado em H1/2024 e avaliado em H2/2024; critérios registrados antes. Melhorou Brier/Log Loss, mas perdeu3,49pp de recall, acima do limite1pp. Portanto não foi adotado. 2024 já foi usado em seleção e não é independente de todas as decisões.

In [2]:
decision=json.loads((OUT/'decisao_calibracao.json').read_text(encoding='utf-8'))
print('Calibração adotada:',decision['adopted'],'Critérios:',decision['checks'])
print('Data da decisão UTC:',decision['decided_at'])
print((OUT/'avaliacao_unica.json').read_text(encoding='utf-8'))


Calibração adotada: False Critérios: {'positive_slope': True, 'brier_gain': True, 'log_loss_gain': True, 'ece10_guard': True, 'ece20_guard': True, 'macro_f1_guard': True, 'recall_guard': False, 'Q3_2024_brier_guard': True, 'Q3_2024_loss_guard': True, 'Q4_2024_brier_guard': True, 'Q4_2024_loss_guard': True}
Data da decisão UTC: 2026-11-09T22:42:05.972593+00:00
{
  "status": "completed",
  "evaluation_id": "gs-celulares-sp-dense64-v1.0.0-teste-2025-unico",
  "started_at": "2026-11-09T22:46:11.641021+00:00",
  "manifest_sha256": "b992412335c69623bdf63c9341cefaa82e0a81ae73d7d32e110f2498faa8c532",
  "frozen_at": "2026-11-09T22:42:06.159595+00:00",
  "pretests_sha256": "04f3870cc7640fa642c6d8254b1275f2162e3a86439995dc03cbfad6ae70e85f",
  "prediction_calls": 1,
  "test_year": 2025,
  "model_version": "gs-celulares-sp-dense64-v1.0.0",
  "evaluation_code_sha256": "5fd4ec2290eca26cb9fcb40c2bd06c44e22d3c0757bbcd523bca31564c555146",
  "metrics_code_sha256": "7f02c89489a6741458f52df8058cb938ef4de6e

![Calibração de desenvolvimento](../reports/etapa5/calibracao_desenvolvimento_2024.png)

## Teste independente

Somente após hashes e75 testes prévios aprovados, os156.168 BOs de2025 foram avaliados uma vez. Nenhuma arquitetura alternativa, calibrador ou threshold foi comparado no teste. A referência usa a classe majoritária do treino2023. Os IC95% usam10.000 réplicas IID da distribuição conjunta(y,pred), seed12345; condicionais ao modelo e sem incerteza de treinamento ou drift.

In [3]:
print((OUT/'metricas_2025.json').read_text(encoding='utf-8'))
print((OUT/'intervalos_confianca_2025.json').read_text(encoding='utf-8'))
print((OUT/'referencia_majoritaria_2025.json').read_text(encoding='utf-8'))


{
  "n": 156168,
  "furtos": 99607,
  "roubos": 56561,
  "tn": 71598,
  "fp": 28009,
  "fn": 20691,
  "tp": 35870,
  "accuracy": 0.6881563444495671,
  "precision_roubo": 0.5615303934000219,
  "recall_roubo": 0.6341825639574972,
  "f1_roubo": 0.5956492859515111,
  "precision_furto": 0.7758020999252349,
  "recall_furto": 0.7188049032698505,
  "f1_furto": 0.7462167007128861,
  "macro_f1": 0.6709329933321986,
  "balanced_accuracy": 0.6764937336136738,
  "brier": 0.2005471646975279,
  "log_loss": 0.5835064622407448,
  "high_conf_errors": 458,
  "high_conf_fn": 439,
  "ece10": 0.08096399905311927,
  "ece20": 0.08100878982451672,
  "confusion_matrix": [
    [
      71598,
      28009
    ],
    [
      20691,
      35870
    ]
  ]
}
{
  "method": {
    "method": "IID nao parametrico das categorias conjuntas (y,pred), equivalente a reamostrar BOs para metricas de matriz",
    "replicates": 10000,
    "seed": 12345,
    "CI_percentiles": [
      2.5,
      97.5
    ]
  },
  "metrics": {
    "ac

![Matriz final](../reports/etapa5/matriz_confusao_2025.png)
![Classes](../reports/etapa5/desempenho_por_classe.png)

## Generalização

Acurácia subiu levemente, mas Macro F1 e precision de roubo caíram. A prevalência de roubo diminuiu. ECE10 subiu de2,71% para8,10%, embora Brier/Log Loss tenham ficado ligeiramente menores; essas medidas não isolam calibração. Padrões de preenchimento de local também mudaram. São hipóteses de mudança de distribuição, não causalidade.

In [4]:
with (OUT/'comparacao_metricas.csv').open(encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):print(row)
print((OUT/'categorias_2025.json').read_text(encoding='utf-8'))


{'metrica': 'accuracy', 'validacao_2024': '0.6850687560967935', 'teste_2025': '0.6881563444495671', 'diferenca_2025_menos_2024': '0.0030875883527735626'}
{'metrica': 'precision_roubo', 'validacao_2024': '0.625754195798961', 'teste_2025': '0.5615303934000219', 'diferenca_2025_menos_2024': '-0.0642238023989391'}
{'metrica': 'recall_roubo', 'validacao_2024': '0.6253141831238779', 'teste_2025': '0.6341825639574972', 'diferenca_2025_menos_2024': '0.008868380833619294'}
{'metrica': 'f1_roubo', 'validacao_2024': '0.6255341120830933', 'teste_2025': '0.5956492859515111', 'diferenca_2025_menos_2024': '-0.029884826131582165'}
{'metrica': 'precision_furto', 'validacao_2024': '0.7280840806931446', 'teste_2025': '0.7758020999252349', 'diferenca_2025_menos_2024': '0.04771801923209029'}
{'metrica': 'recall_furto', 'validacao_2024': '0.7284558144587475', 'teste_2025': '0.7188049032698505', 'diferenca_2025_menos_2024': '-0.009650911188897049'}
{'metrica': 'f1_furto', 'validacao_2024': '0.728269900139554

![Probabilidades](../reports/etapa5/avaliacao_probabilidades.png)
![Meses](../reports/etapa5/desempenho_mensal_2025.png)

## Erros e robustez

Persistem erros nos locais majoritariamente classificados como furto e nos buckets raros/novos. Suportes acompanham as taxas; local ausente tem apenas68 exemplos no teste, sem suporte para generalização. As probabilidades não calibradas não representam risco de crime futuro. Não há rankings de segurança por bairro.

In [5]:
with (OUT/'metricas_grupos_2025.csv').open(encoding='utf-8-sig') as f:
    for row in csv.DictReader(f):
        if row['grupo'] in ['condicao_bairro','presenca_local']:print(row)
for file in ['testes_previos.json','testes_posteriores.json']:
    print(file,(OUT/file).read_text(encoding='utf-8'))


{'grupo': 'condicao_bairro', 'categoria': 'ausente', 'n': '540', 'furtos': '210', 'roubos': '330', 'tn': '73', 'fp': '137', 'fn': '35', 'tp': '295', 'accuracy': '0.6814814814814815', 'precision_roubo': '0.6828703703703703', 'recall_roubo': '0.8939393939393939', 'f1_roubo': '0.7742782152230971', 'precision_furto': '0.6759259259259259', 'recall_furto': '0.3476190476190476', 'f1_furto': '0.4591194968553459', 'macro_f1': '0.6166988560392215', 'balanced_accuracy': '0.6207792207792208', 'brier': '0.22297766621035892', 'log_loss': '0.6409721216178658', 'high_conf_errors': '0', 'high_conf_fn': '0', 'ece10': '0.07217164892289377', 'ece20': '0.0783043045964506', 'confusion_matrix': '[[73, 137], [35, 295]]', 'destacar': 'True'}
{'grupo': 'condicao_bairro', 'categoria': 'desconhecido', 'n': '389', 'furtos': '163', 'roubos': '226', 'tn': '37', 'fp': '126', 'fn': '10', 'tp': '216', 'accuracy': '0.6503856041131105', 'precision_roubo': '0.631578947368421', 'recall_roubo': '0.9557522123893806', 'f1_rou

## Artefato de inferência e conclusão

FinalPipeline aceita exatamente mês, dia, tipo de local e bairro, com as mesmas transformações e avisos estruturados para ausências/raros/novos. A versão consta do manifesto; nenhuma classificação de2025 é recalculada neste notebook. Ver `../docs/contrato_inferencia.md`.

Domínio histórico restrito a celulares subtraídos da capital; não determina natureza jurídica, não prevê crimes, não é sistema oficial e não foi validado para uso policial/operacional. Os resultados finais não alteraram o pipeline. Ver `../reports/relatorio_etapa5.md` e `../docs/recomendacoes_etapa6.md`. Nenhuma aplicação web foi desenvolvida.